# Laboratorio 3.7 (integrador) — Caso conceptual de principio a fin: predicción de churn

**Módulo 3 · Herramientas y Tecnologías** — bloque [`07-casos-conceptuales-y-cierre.md`](../../../Apuntes-Markdown/03-herramientas-y-tecnologias/07-casos-conceptuales-y-cierre.md)

**Duración orientativa:** 150 minutos · **Modalidad:** individual o en parejas, cierre de módulo · **Herramientas:** Google Colab + pandas + scikit-learn

---

Este es el laboratorio de cierre del módulo, y recorre el pipeline completo de un proyecto de datos de principio a fin: limpieza, modelado, evaluación y comunicación del resultado. Todo el código está escrito y verificado; podéis ejecutarlo de principio a fin en Colab.

**Nota para el formador/a:** este laboratorio, junto con la presentación de negocio del entregable, puede usarse como **evaluación final del curso completo**: integra técnicas de los laboratorios 3.1 (EDA), 3.4 (clasificación, métricas, umbral de decisión) y 3.5 (comunicación visual de resultados).

## Objetivo de aprendizaje

Recorrer el pipeline completo de un proyecto de datos — limpieza, modelado, evaluación y comunicación del resultado — replicando el caso conceptual de predicción de churn presentado en el módulo, y practicando el lenguaje problema → features → target → modelo → métricas.

## Contexto

El apunte del bloque 7 dedica su apartado inicial al caso conceptual de predicción de churn, y plantea tres decisiones que son de negocio, no solo técnicas: **qué cuenta como churn** (¿solo cancelación explícita, o también inactividad?), **qué features construir** (ventanas temporales, tendencias, señales de sentimiento) y **qué umbral de clasificación usar** (el coste de no detectar a un cliente que se va puede ser muy superior al de ofrecer un descuento innecesario a alguien que iba a quedarse).

Vais a trabajar con `churn_telecom.csv`, un dataset de 4.200 clientes de una operadora de telecomunicaciones donde `churn=1` significa que el cliente **canceló explícitamente su contrato** (no incluye inactividad, que sería una definición distinta y más ambigua de "abandono"). El dataset tiene una tasa de churn del 20.4%, y contiene correlaciones realistas con la teoría de negocio de las telecos: más churn entre clientes con contrato mes a mes, cargo mensual alto, muchas incidencias de soporte, sin soporte técnico contratado y con poca antigüedad — todo lo que vais a descubrir vosotros mismos con el EDA y confirmar con el modelo.

## Dataset

`churn_telecom.csv` (en esta misma carpeta), 4.200 filas con las columnas: `customer_id, genero, mayor_65, tiene_pareja, tiene_dependientes, antiguedad_meses, tipo_contrato, facturacion_electronica, metodo_pago, servicio_internet, seguridad_online, soporte_tecnico, streaming_tv, cargo_mensual, cargo_total, num_incidencias_soporte, churn` (variable objetivo, 0/1).


## Parte 1 — Limpieza y exploración de los datos (30 min)

Reutilizamos las técnicas del laboratorio 3.1 (`info()`, `describe()`, nulos, `value_counts()`) para conocer el dataset antes de modelar.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 25)
np.random.seed(42)

df = pd.read_csv("churn_telecom.csv")
print(f"Dataset: {df.shape[0]} clientes, {df.shape[1]} columnas")
df.head()

In [ ]:
df.info()

In [ ]:
# Comprobación de calidad de datos: ¿hay valores nulos?
print("Valores nulos por columna:")
print(df.isna().sum().sum(), "nulos en total")
df.describe()

### ¿Qué es "churn" en este dataset?

Antes de seguir, dejamos por escrito la definición del target, tal como recomienda el apunte ("la definición del target es una decisión de negocio, no técnica"):

> **`churn = 1`** significa que el cliente **canceló explícitamente su contrato** con la operadora. No incluye clientes inactivos que aún no han cancelado formalmente — esa sería una definición distinta (y más difícil de medir, porque requeriría definir un umbral de inactividad) que no es la que usa este dataset.

In [ ]:
tasa_churn = df["churn"].mean()
print(f"Tasa de churn global: {tasa_churn:.1%}")
print()
print("Distribución de churn:")
print(df["churn"].value_counts())
print()
print("Distribución por variable de contrato (value_counts):")
print(df["tipo_contrato"].value_counts())

In [ ]:
# Exploramos cómo varía la tasa de churn según algunas variables clave, reutilizando
# groupby().agg() del laboratorio 3.1
tasa_por_contrato = df.groupby("tipo_contrato")["churn"].agg(["mean", "count"]).rename(
    columns={"mean": "tasa_churn", "count": "num_clientes"}
).sort_values("tasa_churn", ascending=False)
tasa_por_contrato["tasa_churn"] = (tasa_por_contrato["tasa_churn"] * 100).round(1)
print("Tasa de churn (%) por tipo de contrato:")
tasa_por_contrato

In [ ]:
tasa_por_soporte = df.groupby("soporte_tecnico")["churn"].agg(["mean", "count"]).rename(
    columns={"mean": "tasa_churn", "count": "num_clientes"}
)
tasa_por_soporte["tasa_churn"] = (tasa_por_soporte["tasa_churn"] * 100).round(1)
print("Tasa de churn (%) según si el cliente tiene soporte técnico contratado:")
tasa_por_soporte

**Lo que deberíais observar:** los clientes con contrato "Mes a mes" tienen una tasa de churn muy superior (en torno al 28%) a los de "Dos años" (en torno al 9%), y los clientes sin soporte técnico contratado abandonan algo más que los que sí lo tienen. Esto confirma, con datos, la intuición de negocio del apunte: la falta de compromiso a largo plazo (contrato corto) y la falta de acompañamiento (sin soporte técnico) se asocian a más abandono.

## Parte 2 — Feature engineering (20 min)

Construimos dos variables derivadas, siguiendo la misma lógica de feature engineering del laboratorio 3.1 (apunte: ratios y variables booleanas derivadas de un umbral):

1. **`cargo_mensual_alto`**: variable booleana que indica si el cargo mensual del cliente supera la mediana de todos los clientes.
2. **`cargo_por_mes_antiguedad`**: ratio entre el cargo total acumulado y la antigüedad en meses — una forma de normalizar el gasto total por el tiempo que lleva el cliente, útil para comparar clientes con antigüedades muy distintas.

In [ ]:
mediana_cargo_mensual = df["cargo_mensual"].median()
print(f"Mediana de cargo_mensual: {mediana_cargo_mensual:.2f} EUR")

df["cargo_mensual_alto"] = (df["cargo_mensual"] > mediana_cargo_mensual).astype(int)

# Evitamos división por cero para clientes con antiguedad_meses == 0 (clientes nuevos
# que se dieron de baja el mismo mes de alta, un caso límite real de negocio)
df["cargo_por_mes_antiguedad"] = df["cargo_total"] / df["antiguedad_meses"].replace(0, 1)

df[["cargo_mensual", "cargo_mensual_alto", "cargo_total", "antiguedad_meses", "cargo_por_mes_antiguedad"]].head()

In [ ]:
# Verificación rápida de la nueva feature booleana: ¿tiene más churn el grupo de cargo alto?
tasa_por_cargo_alto = df.groupby("cargo_mensual_alto")["churn"].mean() * 100
print("Tasa de churn (%) según cargo_mensual_alto (0 = por debajo de la mediana, 1 = por encima):")
print(tasa_por_cargo_alto.round(1))

## Parte 3 — Modelado y evaluación completa (40 min)

Entrenamos un Random Forest (aprovechamos que ofrece `.feature_importances_`, útil para el Gráfico 2 de la comunicación de resultados) con un `Pipeline` que incluye el preprocesado, siguiendo exactamente la misma metodología del laboratorio 3.4: separación train/test, `ColumnTransformer` con escalado de numéricas y one-hot de categóricas, y evaluación con matriz de confusión, precision, recall y F1.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, ConfusionMatrixDisplay, precision_score,
    recall_score, f1_score, roc_auc_score,
)

columnas_numericas = ["antiguedad_meses", "cargo_mensual", "cargo_total",
                       "num_incidencias_soporte", "cargo_por_mes_antiguedad"]
columnas_binarias = ["mayor_65", "cargo_mensual_alto"]
columnas_categoricas = ["genero", "tiene_pareja", "tiene_dependientes", "tipo_contrato",
                         "facturacion_electronica", "metodo_pago", "servicio_internet",
                         "seguridad_online", "soporte_tecnico", "streaming_tv"]

columnas_features = columnas_numericas + columnas_binarias + columnas_categoricas
X = df[columnas_features]
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f"Train: {len(X_train)} clientes ({y_train.mean():.1%} churn)")
print(f"Test:  {len(X_test)} clientes ({y_test.mean():.1%} churn)")

In [ ]:
preprocesador = ColumnTransformer(transformers=[
    ("numericas", StandardScaler(), columnas_numericas),
    ("binarias", "passthrough", columnas_binarias),
    ("categoricas", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas),
])

pipeline_churn = Pipeline(steps=[
    ("preprocesado", preprocesador),
    ("modelo", RandomForestClassifier(
        n_estimators=300, max_depth=6, class_weight="balanced",
        random_state=42, n_jobs=-1,
    )),
])

pipeline_churn.fit(X_train, y_train)
print("Modelo de churn entrenado.")

In [ ]:
y_pred_test = pipeline_churn.predict(X_test)
y_proba_test = pipeline_churn.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_test, target_names=["No churn", "Churn"], digits=3))
print(f"ROC-AUC en test: {roc_auc_score(y_test, y_proba_test):.3f}")

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_test, display_labels=["No churn", "Churn"], cmap="Blues",
)
plt.title("Matriz de confusión — predicción de churn (umbral 0.5)")
plt.tight_layout()
plt.show()

### Importancia de variables

Con `.feature_importances_` extraemos qué variables pesan más en las decisiones del Random Forest — la explicación global del apunte del bloque 4 ("Interpretabilidad y Explicabilidad").

In [ ]:
nombres_columnas_transformadas = pipeline_churn.named_steps["preprocesado"].get_feature_names_out()
importancias = pipeline_churn.named_steps["modelo"].feature_importances_

tabla_importancias = pd.DataFrame({
    "variable": nombres_columnas_transformadas,
    "importancia": importancias,
}).sort_values("importancia", ascending=False).head(10).reset_index(drop=True)

tabla_importancias

## Parte 4 — Comunicar el resultado a una audiencia de negocio (40 min)

Elaboramos 2-3 gráficos pensados para una audiencia **no técnica** (dirección, equipo comercial), no para un informe técnico: pocos elementos, etiquetas claras, un mensaje por gráfico.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

tasa_por_contrato_pct = df.groupby("tipo_contrato")["churn"].mean().sort_values(ascending=False) * 100

barras = ax.bar(tasa_por_contrato_pct.index, tasa_por_contrato_pct.values,
                 color=["#C44E52", "#DD8452", "#4C72B0"])
ax.set_ylabel("Tasa de churn (%)")
ax.set_title("Gráfico 1 — El tipo de contrato es el factor más determinante del abandono")
ax.set_ylim(0, tasa_por_contrato_pct.max() * 1.25)

for barra, valor in zip(barras, tasa_por_contrato_pct.values):
    ax.text(barra.get_x() + barra.get_width() / 2, valor + 1, f"{valor:.1f}%",
            ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("grafico_1_churn_por_contrato.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

top_variables = tabla_importancias.head(6).iloc[::-1]  # invertido para que la barra más importante quede arriba
ax.barh(top_variables["variable"], top_variables["importancia"], color="#4C72B0")
ax.set_xlabel("Importancia relativa")
ax.set_title("Gráfico 2 — Variables que más influyen en la predicción de churn")

plt.tight_layout()
plt.savefig("grafico_2_importancia_variables.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

ax.hist(y_proba_test[y_test == 0], bins=20, alpha=0.6, label="Clientes que NO se fueron", color="#4C72B0")
ax.hist(y_proba_test[y_test == 1], bins=20, alpha=0.6, label="Clientes que SÍ se fueron (churn real)", color="#C44E52")
ax.axvline(0.5, color="black", linestyle="--", linewidth=1, label="Umbral por defecto (0.5)")
ax.set_xlabel("Probabilidad de churn estimada por el modelo")
ax.set_ylabel("Número de clientes (conjunto de test)")
ax.set_title("Gráfico 3 — El modelo separa razonablemente bien ambos grupos")
ax.legend()

plt.tight_layout()
plt.savefig("grafico_3_distribucion_probabilidad.png", dpi=150)
plt.show()

### Umbral de decisión: recomendación justificada por el coste relativo de errores

Igual que en el laboratorio 3.4, razonamos el umbral en términos de negocio, no solo de F1:

- **Falso positivo** (contactar a un cliente que en realidad no se iba a ir): coste bajo — una llamada o un descuento ofrecido de más.
- **Falso negativo** (no detectar a un cliente que sí se va): coste alto — se pierde al cliente y todo su valor futuro, sin oportunidad de retenerlo.

Esta asimetría de costes justifica **priorizar el recall por encima de la precision**, pero sin destrozarla del todo (un umbral demasiado bajo generaría tantas alertas que el equipo comercial no podría atenderlas todas, exactamente como en el laboratorio 3.4). Probamos varios umbrales candidatos y comparamos con el 0.5 por defecto.

In [ ]:
filas_umbral_churn = []
for umbral_candidato in [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6]:
    predicciones_umbral = (y_proba_test >= umbral_candidato).astype(int)
    filas_umbral_churn.append({
        "umbral": umbral_candidato,
        "precision": precision_score(y_test, predicciones_umbral),
        "recall": recall_score(y_test, predicciones_umbral),
        "f1": f1_score(y_test, predicciones_umbral),
        "pct_clientes_marcados": predicciones_umbral.mean() * 100,
    })

tabla_umbrales_churn = pd.DataFrame(filas_umbral_churn).round(3)
tabla_umbrales_churn

**Cómo leer esta tabla:** el F1 máximo está en torno al umbral 0.5-0.55, pero ahí el recall (~60%) todavía deja escapar a 4 de cada 10 clientes que sí van a irse. Bajando el umbral a **0.45**, el recall sube a cerca del 70% (detectamos 7 de cada 10 abandonos reales) a cambio de una caída moderada de precision (de ~42% a ~38%) y un volumen de clientes marcados que pasa del 31% al 38% de la base — un incremento notable, pero todavía representa una campaña de retención dirigida y no un contacto masivo indiscriminado a toda la cartera. Es el umbral que recomendaríamos como punto de partida para generar la lista de clientes a contactar de forma proactiva con ofertas de retención, ajustable después según la capacidad real del equipo comercial.

In [ ]:
umbral_recomendado_churn = 0.45  # <-- parámetro que podéis modificar para explorar el compromiso

predicciones_finales = (y_proba_test >= umbral_recomendado_churn).astype(int)
print(f"Regresión con umbral = {umbral_recomendado_churn}:")
print(classification_report(y_test, predicciones_finales, target_names=["No churn", "Churn"], digits=3))

## Entregable

Este notebook ejecutado de principio a fin, más una breve presentación de negocio siguiendo la plantilla de `plantilla-presentacion-negocio.md` (en esta carpeta), que reutiliza los tres gráficos generados en la Parte 4 (guardados como `grafico_1_churn_por_contrato.png`, `grafico_2_importancia_variables.png` y `grafico_3_distribucion_probabilidad.png`).

## Preguntas de reflexión

1. El apunte menciona features basadas en "ventanas temporales" (uso de los últimos 30 días) que este dataset no incluye, porque es una fotografía estática de cada cliente. Si tuvierais acceso al histórico mes a mes de cada cliente, ¿qué feature de ventana temporal añadiríais primero, y por qué creéis que sería predictiva?
2. Comparad el umbral recomendado en este laboratorio (0.45, para churn) con el del laboratorio 3.4 (0.8, para fraude). ¿Por qué tiene sentido que sean tan distintos, aunque en ambos casos se prioriza el recall?
3. Si tuvierais que presentar el Gráfico 3 (distribución de probabilidades) a un director comercial sin formación técnica, ¿cómo explicaríais en una frase qué significa que las dos distribuciones se solapen parcialmente?
